In [ ]:
import pickle

import matplotlib
import matplotlib.pyplot as plt
import os
import mne
import numpy as np
import pandas as pd
import torch
import copy

from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data
import matplotlib.pylab as pylab
import gc
import quantus
from tqdm import tqdm
from captum.attr import GradientShap

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
def load_explanations_gradshap(subject_index=2):
    load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    return gradshap

In [ ]:
def load_explanations_saliency(subject_index=2):
    load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/saliency_explanations"

    saliency = np.load(os.path.join(load_path, f"saliency_explanations_subject_{subject_index}.npy"), allow_pickle=True)
    return saliency



In [ ]:
def load_ch_names(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)

    file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
    epochs = mne.read_epochs(file_path)
    info_subj = epochs.info

    return ch_names, info_subj

In [ ]:
def get_channel_importances(explanations, ch_names, take_abs=False):


    explanations_normed = np.zeros_like(explanations)
    for trial_idx, trial in enumerate(explanations):
        trial = trial / np.linalg.norm(trial)
        explanations_normed[trial_idx] = trial
    
    channel_importances = np.zeros(len(ch_names))

    for trial_idx, trial_normed in enumerate(explanations_normed):
        if take_abs:
            channel_importances += np.mean(np.abs(trial_normed), axis=1)
        else:
            channel_importances += np.mean(trial_normed, axis=1)
    
    channel_importances_dic = {ch_names[i]: channel_importances[i] for i in range(len(ch_names))}

    return channel_importances_dic

In [ ]:
def create_index_groups(n_samples, group_size=100):
    """
    Create index groups for a given subject based on uncertainty values.
    
    Parameters:
    -----------
    uncertainties : array-like
        The uncertainties array for the subject
    subject_index : int
        Index of the subject
    group_size : int, default=100
        Size of each group
    
    Returns:
    --------
    dict
        Dictionary with subject_index as key and array of boolean index groups as value
    """
    #index_groups_all = {}
    index_groups_subject = []
    
    #max_samples = n_samples - group_size
    start = 0
    while start < n_samples:
        end = min(start + group_size, n_samples)
        
        index_group = np.zeros(n_samples, dtype=bool)
        index_group[start:end] = True
        
        if np.sum(index_group) > 1:
            index_groups_subject.append(index_group)
        
        start += group_size
    

    return index_groups_subject

In [ ]:
def load_all_subject_results(subject_indices):
    results = {}
    for subject_index in subject_indices:
        results[subject_index]  = np.load(f'subject_{subject_index}_results.pkl', allow_pickle=True)
    return results

# List of subject indices to load
cfg = load_config()
subject_indices = cfg.dataset.test_subject_indices

# Load results for each subject
#subjects_dicts = load_all_subject_results(subject_indices)

In [ ]:
def get_channel_importances_time_indexed(explanations, ch_names, index_groups, take_abs=False, agg_func="mean"):
    # adapt 
    explanations_normed = np.zeros_like(explanations)
    for trial_idx, trial in enumerate(explanations):
        trial_normed = trial / np.linalg.norm(trial)
        explanations_normed[trial_idx] = trial_normed

    #channel_importances = np.zeros(( len(index_groups),len(ch_names)))
    channel_importances = []
    for i,index_group in enumerate(index_groups):
        explanations_index_group = explanations_normed[index_group]
        channel_importances_index_group = np.zeros(len(ch_names))
        
        for trial_idx, trial_normed in enumerate(explanations_index_group):
            if agg_func == "mean":
                if take_abs:
                # taking the mean within a trial is fine as different time poins are supposed to be more important than other
                    channel_importances_index_group += np.mean(np.abs(trial_normed), axis=1)
                else:
                    channel_importances_index_group += np.mean(trial_normed, axis=1)
            elif agg_func == "median":
                if take_abs:
                    channel_importances_index_group += np.median(np.abs(trial_normed), axis=1)
                else:
                    channel_importances_index_group += np.median(trial_normed, axis=1)
    
        channel_importances_dic = {ch_names[i]: channel_importances_index_group[i] for i in range(len(ch_names))}
        channel_importances.append(channel_importances_dic)
    

    return channel_importances
    

# single subject results

In [ ]:
explanatations_gradshap = load_explanations_gradshap(2)
explanations_saliency = load_explanations_saliency(2)
index_groups = create_index_groups(len(explanatations_gradshap), group_size=100)
ch_names, info = load_ch_names(2)

In [ ]:
channel_importances = get_channel_importances_time_indexed(explanatations_gradshap, ch_names, index_groups, take_abs=False)
channel_importances_abs = get_channel_importances_time_indexed(explanatations_gradshap, ch_names, index_groups, take_abs=True)

In [ ]:

import matplotlib.ticker as ticker
def plot_time_indexed_topomap(channel_importances, ch_names, info, index_groups):
    fig, axs = plt.subplots(1, len(index_groups), figsize=(16, 5))
    fig.subplots_adjust(wspace=0.08)
    for i,ax in zip(range(len(index_groups)), axs):
        channel_importances_index_group = channel_importances[i]
        topomap_data = np.zeros(len(ch_names))
        for ch_idx, ch_name in enumerate(ch_names):
            topomap_data[ch_idx] = channel_importances_index_group[ch_name]
        topo,_ = mne.viz.plot_topomap(topomap_data, info, show=False, axes=ax)

        cb = plt.colorbar(topo, ax=ax, location='bottom', pad=0.05)
        #cb.set_label('Median Difference', fontsize=14)
        cb.ax.tick_params(labelsize=11)
        cb.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
        #ax.set_title(f"Topomap for group {i}")
    fig.savefig("topomaps_time_indexed_gradshap_subject_2.png", bbox_inches='tight')


In [ ]:
plot_time_indexed_topomap(channel_importances, ch_names, info, index_groups)

In [ ]:
plot_time_indexed_topomap(channel_importances_abs, ch_names, info, index_groups)

In [ ]:
channel_importances = get_channel_importances_time_indexed(explanations_saliency, ch_names, index_groups, take_abs=False)
channel_importances_abs = get_channel_importances_time_indexed(explanations_saliency, ch_names, index_groups, take_abs=True)

In [ ]:
plot_time_indexed_topomap(channel_importances, ch_names, info, index_groups)

In [ ]:
plot_time_indexed_topomap(channel_importances_abs, ch_names, info, index_groups)

In [ ]:
explanatations_gradshap = load_explanations_gradshap(67)

index_groups = create_index_groups(len(explanatations_gradshap), group_size=100)
ch_names, info = load_ch_names(67)
channel_importances = get_channel_importances_time_indexed(explanatations_gradshap, ch_names, index_groups, take_abs=False)
channel_importances_abs = get_channel_importances_time_indexed(explanatations_gradshap, ch_names, index_groups, take_abs=True)

In [ ]:
plot_time_indexed_topomap(channel_importances, ch_names, info, index_groups)

In [ ]:
plot_time_indexed_topomap(channel_importances_abs, ch_names, info, index_groups)

In [ ]:
explanations_saliency = load_explanations_saliency(67)
channel_importances = get_channel_importances_time_indexed(explanations_saliency, ch_names, index_groups, take_abs=False)
channel_importances_abs = get_channel_importances_time_indexed(explanations_saliency, ch_names, index_groups, take_abs=True)

In [ ]:
plot_time_indexed_topomap(channel_importances, ch_names, info, index_groups)

In [ ]:
plot_time_indexed_topomap(channel_importances_abs, ch_names, info, index_groups)

# averaged over subjects

## absolute values

In [ ]:
index_groups = create_index_groups(350, group_size=50)

In [ ]:
importances_all_subjects = {subject_index: {} for subject_index in cfg.dataset.test_subject_indices}
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    print(f"Subject {subject_index}")
    ch_names, info = load_ch_names(subject_index)
    explanations_gradshap = load_explanations_gradshap(subject_index)
    index_groups = create_index_groups(len(explanations_gradshap), group_size=72)
    #channel_importances_gradshap = get_channel_importances_time_indexed(explanations_gradshap, ch_names, index_groups, take_abs=False)
    channel_importances_gradshap_abs = get_channel_importances_time_indexed(explanations_gradshap, ch_names, index_groups, take_abs=True)
    importances_all_subjects[subject_index] = channel_importances_gradshap_abs



In [ ]:
def get_common_channels():
    for subject_index in cfg.dataset.test_subject_indices:
        ch_names, info = load_ch_names(subject_index)
        if subject_index == cfg.dataset.test_subject_indices[0]:
            common_channels = ch_names
        else:
            common_channels = list(set(common_channels).intersection(ch_names))
    return common_channels

In [ ]:
common_channels = get_common_channels()

In [ ]:
len(importances_all_subjects[1])

In [ ]:
def plot_averaged_topomaps(importances_all_subjects, common_channels, info, index_groups):
    
    cfg = load_config()
    # only as many topoplots can be made as there a groups for the subject with the least trials
    min_groups = np.inf
    for subject_index in cfg.dataset.test_subject_indices:
        min_groups = min(min_groups, len(importances_all_subjects[subject_index]))

    fig, axs = plt.subplots(1, min_groups, figsize=(16, 5))
    fig.subplots_adjust(wspace=0.08)

    for i,ax in zip(range(min_groups), axs):
        topomap_data = np.zeros(60)
        for subject_index in importances_all_subjects.keys():
            ch_names, info = load_ch_names(subject_index)
            channel_importances_index_group = importances_all_subjects[subject_index][i]
            # subjects have the same number of channels but slightly different channel names, therefore only iterate over common channels (59 out of 60)
            for ch_idx, ch_name in enumerate(common_channels):
                topomap_data[ch_names.index(ch_name)] += channel_importances_index_group[ch_name]
        topomap_data /= len(importances_all_subjects)
        topo,_ = mne.viz.plot_topomap(topomap_data, info, show=False, axes=ax)

        cb = plt.colorbar(topo, ax=ax, location='bottom', pad=0.05)
        #cb.set_label('Median Difference', fontsize=14)
        cb.ax.tick_params(labelsize=11)
        cb.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
        #ax.set_title(f"Topomap for group {i}")
    fig.savefig("topomaps_time_indexed_gradshap_subjects_avg.png", bbox_inches='tight')

In [ ]:
import mne
mne.set_log_level('ERROR')

In [ ]:
plot_averaged_topomaps(importances_all_subjects, common_channels, info, index_groups)

In [ ]:
importances_all_subjects[1][0].values()

In [ ]:
from scipy.stats import spearmanr
import itertools
def compute_rank_correlations_for_each_index_group(importances_all_subjects, common_channels):

    cfg = load_config()
    # only as many topoplots can be made as there a groups for the subject with the least trials
    min_groups = np.inf
    for subject_index in cfg.dataset.test_subject_indices:
        min_groups = min(min_groups, len(importances_all_subjects[subject_index]))

    
    all_rank_correlations = {i: [] for i in range(min_groups)}
    for i in range(min_groups):
        rank_correlations = []
        for s1,s2 in itertools.combinations(cfg.dataset.test_subject_indices, 2):
            #ch_names, info = load_ch_names(s1)
            channel_importances_s1 = np.array(list(importances_all_subjects[s1][i].values()))
            channel_importances_s2 = np.array(list(importances_all_subjects[s2][i].values()))
            rank_correlations.append(spearmanr(channel_importances_s1, channel_importances_s2).correlation)
        all_rank_correlations[i] = rank_correlations

    return all_rank_correlations

In [ ]:
result = compute_rank_correlations_for_each_index_group(importances_all_subjects, common_channels)

In [ ]:
[np.mean(result[i]) for i in range(5)]

In [ ]:
[np.std(result[i]) for i in range(5)]

In [ ]:
[i*72 for i in range(6)]

## non absolute values

In [ ]:
importances_all_subjects = {subject_index: {} for subject_index in cfg.dataset.test_subject_indices}
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    print(f"Subject {subject_index}")
    ch_names, info = load_ch_names(subject_index)
    explanations_gradshap = load_explanations_gradshap(subject_index)
    index_groups = create_index_groups(len(explanations_gradshap), group_size=70)
    #channel_importances_gradshap = get_channel_importances_time_indexed(explanations_gradshap, ch_names, index_groups, take_abs=False)
    channel_importances_gradshap = get_channel_importances_time_indexed(explanations_gradshap, ch_names, index_groups, take_abs=False)
    importances_all_subjects[subject_index] = channel_importances_gradshap


In [ ]:
plot_averaged_topomaps(importances_all_subjects, common_channels, info, index_groups)

In [ ]:
importances_all_subjects = {subject_index: {} for subject_index in cfg.dataset.test_subject_indices}
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    print(f"Subject {subject_index}")
    ch_names, info = load_ch_names(subject_index)
    explanations_saliency= load_explanations_saliency(subject_index)
    index_groups = create_index_groups(len(explanations_gradshap), group_size=70)
    #channel_importances_gradshap = get_channel_importances_time_indexed(explanations_gradshap, ch_names, index_groups, take_abs=False)
    channel_importances_saliency = get_channel_importances_time_indexed(explanations_saliency, ch_names, index_groups, take_abs=False)
    importances_all_subjects[subject_index] = channel_importances_saliency